In [1]:
!pip install fastapi uvicorn python-multipart python-dotenv

In [3]:
!pip install nest-asyncio

In [5]:
from fastapi import FastAPI

app=FastAPI()

@app.get("/")
def home():
    return {"message":"AI API is running"}

In [6]:
print(home())

{'message': 'AI API is running'}


In [13]:
import requests

response=requests.get("http://127.0.0.1:8000/")
print(response.status_code)
print(response.text)

200
{"message":"AI API is running"}


In [15]:
response=requests.post(
    "http://127.0.0.1:8001/chat",
    json={"message":"Hello AI"}
)

print(response.status_code)
print(response.json())

200
{'user_message': 'Hello AI', 'response': 'You said: Hello AI'}


In [16]:
!pip install groq python-dotenv

In [17]:
import requests

response=requests.post(
    "http://127.0.0.1:8001/chat",
    json={"message":"What is artificial intelligence?"}
)

print(response.status_code)
print(response.json())

200
{'user_message': 'What is artificial intelligence?', 'response': 'You said: What is artificial intelligence?'}


In [18]:
import requests

response=requests.post(
    "http://127.0.0.1:8001/chat",
    json={"message":"What is artificial intelligence?"}
)

print(response.status_code)
print(response.json())

200
{'user_message': 'What is artificial intelligence?', 'response': 'Artificial intelligence (AI) is a broad field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence. These tasks include:\n\n1. **Learning**: AI systems can learn from data and improve their performance over time.\n2. **Reasoning**: AI systems can make decisions based on logic and reasoning.\n3. **Problem-solving**: AI systems can solve complex problems that require critical thinking.\n4. **Perception**: AI systems can interpret and understand sensory data from the environment.\n5. **Communication**: AI systems can understand and generate human language.\n\nAI systems are designed to simulate human intelligence and behavior, but they operate based on algorithms, rules, and data, rather than emotions, intuition, or consciousness. There are several types of AI, including:\n\n1. **Narrow or Weak AI**: Designed to perform a specific task, such a

In [19]:
from fastapi import UploadFile,File

In [20]:
@app.post("/upload")
async def upload_file(file:UploadFile=File(...)):
    if not file.filename.endswith((".pdf",".txt")):
        return {"error":"Only PDF and TXT files are allowed"}

    file_content=await file.read()

    with open(file.filename,"wb") as f:
        f.write(file_content)

    return {
        "filename":file.filename,
        "message":"File uploaded successfully"
    }

In [21]:
files={
    "file":open("documents/ai.txt","rb")
}

response=requests.post(
    "http://127.0.0.1:8001/upload",
    files=files
)

print(response.status_code)
print(response.json())

200
{'filename': 'ai.txt', 'message': 'File uploaded successfully'}


In [22]:
files={
    "file":open("documents/AI_Introduction.pdf","rb")
}

response=requests.post(
    "http://127.0.0.1:8001/upload",
    files=files
)

print(response.status_code)
print(response.json())

200
{'filename': 'AI_Introduction.pdf', 'message': 'File uploaded successfully'}


In [23]:
response=requests.post(
    "http://127.0.0.1:8001/ask",
    json={"question":"What is artificial intelligence?"}
)

print(response.status_code)
print(response.json())

200
{'question': 'What is artificial intelligence?', 'answer': 'Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.\n\nAI technologies are designed to perform a wide range of tasks, from simple calculations and data analysis to complex decision-making and problem-solving. Some common types of AI include:\n\n1. **Narrow or Weak AI**: This type of AI is designed to perform a specific task, such as facial recognition, language translation, or playing chess.\n2. **General or Strong AI**: This type of AI would be capable of performing any intellectual task that a human can, possessing a level of intelligence similar to or exceeding that of the human brain.\n3. **Superintelligence**: This type of AI would significantly surpass the intelligence of the best human minds, potenti

In [28]:
import os

documents_folder="documents"
documents={}

for filename in os.listdir(documents_folder):
    if filename.endswith(".txt"):
        file_path=os.path.join(documents_folder,filename)

        with open(file_path,"r",encoding="utf-8") as file:
            documents[filename]=file.read()

print("Documents loaded:",len(documents))

Documents loaded: 4


In [29]:
chunks=[]

for filename,content in documents.items():
    sentences=content.split(". ")

    current_chunk=""

    for sentence in sentences:
        if len(current_chunk)+len(sentence)<500:
            current_chunk+=sentence+". "
        else:
            chunks.append({
                "file":filename,
                "text":current_chunk.strip()
            })
            current_chunk=sentence+". "

    if current_chunk:
        chunks.append({
            "file":filename,
            "text":current_chunk.strip()
        })

print("Total chunks:",len(chunks))

Total chunks: 8


In [25]:
import chromadb

chroma_client=chromadb.PersistentClient(path="chroma_db")

collection=chroma_client.get_or_create_collection(
    name="documents"
)

In [30]:
from sentence_transformers import SentenceTransformer

embedding_model=SentenceTransformer("all-MiniLM-L6-v2")

texts=[chunk["text"] for chunk in chunks]

embeddings=embedding_model.encode(texts)

print("Generated embeddings:",len(embeddings))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generated embeddings: 8


In [31]:
import chromadb

chroma_client=chromadb.PersistentClient(path="chroma_db")

collection=chroma_client.get_or_create_collection(
    name="documents"
)

ids=[str(i) for i in range(len(chunks))]

documents_list=[chunk["text"] for chunk in chunks]

metadatas=[
    {"source":chunk["file"]}
    for chunk in chunks
]

collection.add(
    ids=ids,
    documents=documents_list,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Vectors stored successfully!")
print(f"Total Documents: {collection.count()}")

Vectors stored successfully!
Total Documents: 8


In [33]:
response=requests.post(
    "http://127.0.0.1:8001/ask",
    json={"question":"What is artificial intelligence?"}
)

print(response.status_code)
print(response.json())

200
{'question': 'What is artificial intelligence?', 'answer': 'Artificial Intelligence (AI) is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence.', 'sources': [{'source': 'ai.txt'}, {'source': 'machine_learning.txt'}, {'source': 'deep_learning.txt'}]}


In [34]:
response=requests.post(
    "http://127.0.0.1:8001/chat",
    json={}
)

print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'missing', 'loc': ['body', 'message'], 'msg': 'Field required', 'input': {}}]}


In [35]:
files={
    "file":open("Week-03_Task#06.ipynb","rb")
}

response=requests.post(
    "http://127.0.0.1:8001/upload",
    files=files
)

print(response.status_code)
print(response.json())

200
{'error': 'Only PDF and TXT files are allowed'}
